# Clase 6 — Mapas interactivos con Folium

## ¿Por qué te importa esto?

Cuando tus datos tienen una ubicación geográfica —una dirección, unas coordenadas, un país— una tabla o un gráfico de barras esconde la parte más obvia de la historia: *dónde* pasan las cosas. Si tienes 100 incidentes policiales con latitud y longitud, o inmigración a Canadá desglosada por país, un mapa muestra patrones espaciales que ningún otro gráfico puede mostrar de forma tan directa.

## La analogía

Folium es como planear un viaje sobre un mapa físico: coges un mapa real (no un dibujo abstracto de ejes X e Y) y vas clavando alfileres de colores en los lugares que te interesan. Cada alfiler puede tener una etiqueta ("aquí hay un robo", "aquí hay un café"). Si tienes demasiados alfileres juntos, los agrupas en un número que dice "aquí hay 40 puntos" hasta que te acerques lo suficiente para verlos uno a uno. Y si en vez de puntos quieres colorear regiones enteras (países, estados) según un valor —más oscuro donde el valor es más alto—, eso es un mapa **coroplético**: como colorear un mapa político según la temperatura o la población de cada país.

Folium hace todo esto sobre mapas reales e interactivos (puedes hacer zoom y desplazarte), a diferencia de Matplotlib, que dibuja sobre un lienzo abstracto.


## 0. Instalación e importación de librerías

In [ ]:
!pip install folium


In [ ]:
import numpy as np
import pandas as pd
import folium

print('Folium instalado e importado!')


## 1. Crear un mapa base

Lo primero es instanciar un objeto `Map`. Sin argumentos, Folium te da el mapa mundial completo.

In [ ]:
# mapa del mundo, sin centro definido
world_map = folium.Map()
world_map


Para centrar el mapa en un lugar concreto, le pasas `location=[latitud, longitud]` y un nivel de zoom con `zoom_start` (cuanto más alto, más acercado).

In [ ]:
# mapa centrado en Canadá, zoom bajo (vista amplia)
world_map = folium.Map(location=[56.130, -106.35], zoom_start=4)
world_map


In [ ]:
# el mismo centro, pero con más zoom (vista más cercana)
world_map = folium.Map(location=[56.130, -106.35], zoom_start=8)
world_map


También puedes cambiar el estilo visual del mapa con el parámetro `tiles`, por ejemplo un estilo oscuro de alto contraste (`'Cartodb dark_matter'`) o uno minimalista claro (`'Cartodb positron'`).

In [ ]:
# estilo oscuro, alto contraste
world_map_dark = folium.Map(location=[56.130, -106.35], zoom_start=4,
                             tiles='Cartodb dark_matter')
world_map_dark


In [ ]:
# estilo claro y minimalista
world_map_light = folium.Map(location=[56.130, -106.35], zoom_start=4,
                              tiles='Cartodb positron')
world_map_light


## 2. Añadir marcadores

Con el mapa base listo, el siguiente paso es "clavar los alfileres". Vamos a usar el dataset de incidentes policiales de San Francisco de 2016 (`df_incidents`), que tiene columnas `Y` (latitud), `X` (longitud) y `Category` (tipo de incidente). Por costo computacional, trabajamos solo con los primeros 100 registros.

Nota: el portal original de datos abiertos de San Francisco (data.sfgov.org) ya no aloja este dataset con esta estructura tras una actualización de sistemas del departamento de policía. Sin embargo, la copia usada en este curso (alojada en el bucket de Skills Network / IBM) sigue disponible públicamente, así que la usamos tal cual.

In [ ]:
df_incidents = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Police_Department_Incidents_-_Previous_Year__2016_.csv')

print('Dataset descargado y leído en un dataframe de pandas!')
df_incidents.head()


In [ ]:
# quedarnos solo con los primeros 100 incidentes (para no saturar el mapa)
limit = 100
df_incidents = df_incidents.iloc[0:limit, :]
df_incidents.shape


In [ ]:
# coordenadas de San Francisco
latitude = 37.77
longitude = -122.42

sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)
sanfran_map


La forma en Folium de superponer varios puntos es crear un **feature group** (grupo de elementos) y añadirle cada marcador.

In [ ]:
# grupo de elementos para los incidentes
incidents = folium.map.FeatureGroup()

for lat, lng in zip(df_incidents.Y, df_incidents.X):
    incidents.add_child(
        folium.vector_layers.CircleMarker(
            [lat, lng],
            radius=5,          # tamaño del círculo
            color='yellow',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6
        )
    )

# añadir el grupo completo al mapa
sanfran_map.add_child(incidents)


Para que cada marcador muestre información al pasar el ratón (un **popup**), añades marcadores de texto en las mismas coordenadas.

In [ ]:
sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)

# grupo de elementos para los incidentes (círculos)
incidents = folium.map.FeatureGroup()
for lat, lng in zip(df_incidents.Y, df_incidents.X):
    incidents.add_child(
        folium.vector_layers.CircleMarker(
            [lat, lng],
            radius=5,
            color='yellow',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6
        )
    )
sanfran_map.add_child(incidents)

# marcadores con popup mostrando la categoría del incidente
latitudes = list(df_incidents.Y)
longitudes = list(df_incidents.X)
labels = list(df_incidents.Category)

for lat, lng, label in zip(latitudes, longitudes, labels):
    folium.Marker([lat, lng], popup=label).add_to(sanfran_map)

sanfran_map


Una alternativa más simple, si el mapa se ve muy recargado con dos capas de marcadores, es poner el `popup` directamente en el `CircleMarker`.

In [ ]:
sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)

for lat, lng, label in zip(df_incidents.Y, df_incidents.X, df_incidents.Category):
    folium.vector_layers.CircleMarker(
        [lat, lng],
        radius=5,
        color='yellow',
        fill=True,
        popup=label,
        fill_color='blue',
        fill_opacity=0.6
    ).add_to(sanfran_map)

sanfran_map


## 3. Agrupar marcadores con MarkerCluster

Con muchos puntos, el mapa se satura visualmente. La solución de Folium es agruparlos en clústeres: cuando estás alejado, ves un número (cuántos incidentes hay en esa zona); cuando te acercas con zoom, el clúster se va abriendo hasta mostrar marcadores individuales.

In [ ]:
from folium import plugins

sanfran_map = folium.Map(location=[latitude, longitude], zoom_start=12)

# objeto MarkerCluster añadido directamente al mapa
incidents = plugins.MarkerCluster().add_to(sanfran_map)

for lat, lng, label in zip(df_incidents.Y, df_incidents.X, df_incidents.Category):
    folium.Marker(
        location=[lat, lng],
        icon=None,
        popup=label,
    ).add_to(incidents)

sanfran_map


Al alejar el zoom del todo, ves un único clúster con el total (100, en este caso). Al acercarte, ese clúster se rompe en clústeres más pequeños hasta llegar a marcadores individuales.

## 4. Choropleth: colorear regiones según un valor

Un mapa **coroplético** no coloca puntos, colorea regiones completas (países, estados, provincias) según un valor numérico: más oscuro (o más rojo, dependiendo de la paleta) donde el valor es más alto. Vamos a construir uno con la inmigración total a Canadá por país, usando `df_can`.

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

print('Datos descargados y leídos en un dataframe!')
df_can.head()


Para el mapa coroplético necesitamos un archivo **GeoJSON** que defina los límites geográficos de cada país (en este caso, `world_countries.json`).

In [ ]:
# GeoJSON con los límites de todos los países del mundo
world_geo = r'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/world_countries.json'

# mapa base centrado en [0, 0] con zoom muy alejado (vista mundial)
world_map = folium.Map(location=[0, 0], zoom_start=2)
world_map


`folium.Choropleth` necesita cuatro cosas clave: el GeoJSON (`geo_data`), el DataFrame con los datos (`data`), qué columnas usar (`columns`: la clave que coincide con el GeoJSON y el valor a colorear), y `key_on`, que indica qué propiedad del GeoJSON contiene el nombre que hace de enlace (en este archivo, `feature.properties.name`, y debe coincidir exactamente, mayúsculas incluidas, con los nombres de país usados en el GeoJSON). En versiones recientes de Folium, `Choropleth` es una capa que se crea aparte y se añade al mapa con `.add_to(mapa)` (el método antiguo `mapa.choropleth(...)` ya no existe en Folium 0.14+).

In [ ]:
folium.Choropleth(
    geo_data=world_geo,
    data=df_can,
    columns=['Country', 'Total'],       # columna de enlace y columna de valor
    key_on='feature.properties.name',   # propiedad del GeoJSON que coincide con 'Country'
    fill_color='YlOrRd',                # paleta de colores (amarillo-naranja-rojo)
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Immigration to Canada'
).add_to(world_map)

world_map

El resultado: cuanto más rojo/oscuro un país, más inmigrantes envió a Canadá entre 1980 y 2013. Con este dataset, los países más oscuros son China, India y Filipinas, seguidos de Polonia, Pakistán y Estados Unidos.

## Puntos clave

- `folium.Map(location=[lat, lon], zoom_start=n)` crea el mapa base; sin `location` obtienes el mapa mundial completo.
- El parámetro `tiles` cambia el estilo visual del mapa (por ejemplo, `'Cartodb dark_matter'` o `'Cartodb positron'`).
- `folium.Marker(...)` y `folium.vector_layers.CircleMarker(...)` añaden puntos al mapa; el parámetro `popup` muestra texto al interactuar con el marcador.
- Un `folium.map.FeatureGroup()` te permite agrupar varios elementos y añadirlos al mapa de una sola vez con `add_child`.
- `plugins.MarkerCluster()` agrupa automáticamente marcadores cercanos en un número que se abre a medida que haces zoom, evitando mapas saturados.
- Un mapa **coroplético** (`folium.Choropleth(...).add_to(mapa)`) colorea regiones completas según un valor, usando un archivo GeoJSON como base de las fronteras y `key_on` para enlazar tus datos con las propiedades del GeoJSON.
- El nombre en `key_on` debe coincidir exactamente (mayúsculas y formato) con el usado en el GeoJSON, o el mapa no coloreará nada.